# NCBI Enzyme Cofactor Data Collection

Queries NCBI for enzyme representatives that use specific cofactors (Cu, ATP, B12, CoA, NAD).
Retrieves sequences and creates binary labels for cofactor usage.
Exports data to CSV format.

**Run all cells from top to bottom** (use "Run All" or execute sequentially)

## Usage Instructions

1. Replace the email address in the import cell with your actual email (required by NCBI)
2. Run cells sequentially from top to bottom (or use "Run All")
3. The script will:
   - Search NCBI for enzymes using Cu, ATP, B12, CoA, NAD
   - Use multiple search strategies to find at least 400 sequences per cofactor
   - Retrieve up to 500 sequences per cofactor
   - Create binary labels (1 = uses cofactor, 0 = doesn't use it)
   - Export to CSV file: `ncbi_enzyme_cofactor_data.csv`

Output CSV format:
- `header`: Protein description/ID
- `sequence`: Amino acid sequence
- `Cu`, `ATP`, `B12`, `CoA`, `NAD`: Binary labels (0 or 1)

Note: NCBI queries may take 20-40 minutes due to multiple search strategies and rate limiting.

In [1]:
# Install required packages for NCBI queries and data processing
%pip install biopython pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Import required libraries

import pandas as pd
from Bio import Entrez, SeqIO
import time
import re
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set your email for NCBI Entrez (required by NCBI)
Entrez.email = "your_email@example.com"  # Replace with your actual email

In [3]:
# Define the 5 cofactors to search for

cofactors = ['Cu', 'ATP', 'B12', 'CoA', 'NAD']

# Minimum number of proteins required per cofactor
MIN_PROTEINS_PER_COFACTOR = 400
MAX_PROTEINS_PER_COFACTOR = 500

print(f"Searching for enzymes that use: {', '.join(cofactors)}")
print(f"Target: {MIN_PROTEINS_PER_COFACTOR}-{MAX_PROTEINS_PER_COFACTOR} sequences per cofactor")
print(f"Total expected: ~{MIN_PROTEINS_PER_COFACTOR * len(cofactors)} - {MAX_PROTEINS_PER_COFACTOR * len(cofactors)} proteins")
print(f"\nThe search will use multiple query strategies to find enough representatives for each cofactor.")

Searching for enzymes that use: Cu, ATP, B12, CoA, NAD
Target: 400-500 sequences per cofactor
Total expected: ~2000 - 2500 proteins

The search will use multiple query strategies to find enough representatives for each cofactor.


In [4]:
# Define functions to search NCBI and fetch protein sequences

def search_ncbi_protein_extensive(search_terms_list, min_results=400, max_total=500):
    """
    Search NCBI Protein database with multiple search strategies until we get enough results.
    Returns list of unique protein IDs.
    """
    all_ids = set()
    
    for search_term in search_terms_list:
        if len(all_ids) >= min_results:
            break
        
        # Try up to 3 times for each query with increasing delays
        for attempt in range(3):
            try:
                # Search the protein database
                handle = Entrez.esearch(
                    db="protein",
                    term=search_term,
                    retmax=max_total,
                    sort="relevance"
                )
                record = Entrez.read(handle)
                handle.close()
                
                id_list = record["IdList"]
                all_ids.update(id_list)
                print(f"    Query '{search_term}': +{len(id_list)} IDs (total unique: {len(all_ids)})")
                
                time.sleep(0.5)  # Rate limiting between queries
                break  # Success, move to next query
                
            except Exception as e:
                if attempt < 2:  # Only retry if not the last attempt
                    wait_time = (attempt + 1) * 2  # 2, 4 seconds
                    print(f"    Query '{search_term}' failed (attempt {attempt+1}/3), retrying in {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    print(f"    Error with query '{search_term}': {e}")
                continue
    
    return list(all_ids)


def search_ncbi_protein(search_term, max_results=20):
    """
    Search NCBI Protein database for a given term.
    Returns list of protein IDs.
    """
    try:
        # Search the protein database
        handle = Entrez.esearch(
            db="protein",
            term=search_term,
            retmax=max_results,
            sort="relevance"
        )
        record = Entrez.read(handle)
        handle.close()
        
        id_list = record["IdList"]
        return id_list
    
    except Exception as e:
        print(f"Error searching NCBI: {e}")
        return []


def fetch_protein_sequences(id_list, batch_size=10):
    """
    Fetch protein sequences from NCBI given a list of protein IDs.
    Returns dict: {protein_id: (header, sequence)}
    Handles errors by retrying failed batches with smaller sizes or individual IDs.
    """
    sequences = {}
    failed_ids = []
    
    # Process in batches to avoid overwhelming the server
    for i in range(0, len(id_list), batch_size):
        batch = id_list[i:i+batch_size]
        
        try:
            # Fetch sequences
            handle = Entrez.efetch(
                db="protein",
                id=batch,
                rettype="fasta",
                retmode="text"
            )
            
            # Parse FASTA records
            for record in SeqIO.parse(handle, "fasta"):
                protein_id = record.id.split('|')[0] if '|' in record.id else record.id
                header = record.description
                sequence = str(record.seq)
                sequences[protein_id] = (header, sequence)
            
            handle.close()
            time.sleep(0.5)  # Rate limiting
            
        except Exception as e:
            # If batch fails, try fetching IDs individually
            print(f"    Batch {i//batch_size + 1} failed, retrying individually...")
            
            for single_id in batch:
                try:
                    handle = Entrez.efetch(
                        db="protein",
                        id=[single_id],
                        rettype="fasta",
                        retmode="text"
                    )
                    
                    for record in SeqIO.parse(handle, "fasta"):
                        protein_id = record.id.split('|')[0] if '|' in record.id else record.id
                        header = record.description
                        sequence = str(record.seq)
                        sequences[protein_id] = (header, sequence)
                    
                    handle.close()
                    time.sleep(0.3)
                    
                except Exception as e2:
                    # Skip this ID if it still fails
                    failed_ids.append(single_id)
                    continue
    
    if failed_ids:
        print(f"    ⚠ Skipped {len(failed_ids)} invalid/inaccessible IDs")
    
    return sequences

In [5]:
# Search NCBI for enzymes that use each cofactor (with multiple search strategies)

print("Searching NCBI for enzymes with each cofactor...\n")
print("Will try multiple search strategies to find at least 400 of each\n")

# Define multiple search strategies for each cofactor (using simpler terms to avoid NCBI backend errors)
cofactor_search_strategies = {
    'Cu': [
        'copper',
        'cupredoxin',
        'copper-binding',
        'Cu protein',
        'copper oxidase',
        'copper enzyme',
        'metallothionein copper'
    ],
    'ATP': [
        'ATPase',
        'ATP-binding',
        'ATP synthase',
        'adenylate kinase',
        'kinase ATP',
        'ATP hydrolase',
        'nucleotide-binding'
    ],
    'B12': [
        'vitamin B12',
        'cobalamin',
        'methylcobalamin',
        'adenosylcobalamin',
        'B12-dependent',
        'cobalamin-binding',
        'vitamin B12 enzyme'
    ],
    'CoA': [
        'CoA-binding protein',
        'acyl-CoA',
        'acetyl-CoA',
        'coenzyme A',
        'CoA ligase',
        'acyl-CoA dehydrogenase',
        'CoA transferase'
    ],
    'NAD': [
        'NAD-dependent',
        'NADH dehydrogenase',
        'NAD-binding',
        'NADH oxidoreductase',
        'NAD+ binding',
        'nicotinamide adenine',
        'NAD enzyme'
    ]
}

cofactor_proteins = {}
MIN_PROTEINS_PER_COFACTOR = 400

for cofactor in cofactors:
    search_terms = cofactor_search_strategies[cofactor]
    print(f"Searching for {cofactor}-binding enzymes:")
    
    # Use extensive search with multiple strategies
    protein_ids = search_ncbi_protein_extensive(
        search_terms, 
        min_results=MIN_PROTEINS_PER_COFACTOR,
        max_total=500
    )
    
    print(f"  ✓ Total unique {cofactor} proteins found: {len(protein_ids)}")
    
    if protein_ids:
        # Take first 500 if we got more than that
        cofactor_proteins[cofactor] = protein_ids[:500]
        time.sleep(1)  # Rate limiting between cofactors
    
    print()

print("="*80)
print("SEARCH COMPLETE")
print("="*80)
print(f"\nTotal cofactors with results: {len(cofactor_proteins)}")
for cf, ids in cofactor_proteins.items():
    status = "✓" if len(ids) >= MIN_PROTEINS_PER_COFACTOR else "⚠"
    print(f"  {status} {cf}: {len(ids)} proteins")

Searching NCBI for enzymes with each cofactor...

Will try multiple search strategies to find at least 400 of each

Searching for Cu-binding enzymes:
    Query 'copper' failed (attempt 1/3), retrying in 2s...
    Query 'copper' failed (attempt 2/3), retrying in 4s...
    Query 'copper': +500 IDs (total unique: 500)
  ✓ Total unique Cu proteins found: 500

Searching for ATP-binding enzymes:
    Query 'ATPase': +500 IDs (total unique: 500)
  ✓ Total unique ATP proteins found: 500

Searching for B12-binding enzymes:
    Query 'vitamin B12': +500 IDs (total unique: 500)
  ✓ Total unique B12 proteins found: 500

Searching for CoA-binding enzymes:
    Query 'CoA-binding protein': +0 IDs (total unique: 0)
    Query 'acyl-CoA': +500 IDs (total unique: 500)
  ✓ Total unique CoA proteins found: 500

Searching for NAD-binding enzymes:
    Query 'NAD-dependent': +500 IDs (total unique: 500)
  ✓ Total unique NAD proteins found: 500

SEARCH COMPLETE

Total cofactors with results: 5
  ✓ Cu: 500 prote

In [6]:
# Fetch sequences for all proteins

print("Fetching sequences from NCBI...\n")

all_sequences = {}

for cofactor, protein_ids in cofactor_proteins.items():
    print(f"Fetching sequences for {cofactor}-binding enzymes ({len(protein_ids)} proteins)...")
    
    sequences = fetch_protein_sequences(protein_ids)
    
    for protein_id, (header, sequence) in sequences.items():
        if protein_id not in all_sequences:
            all_sequences[protein_id] = {
                'header': header,
                'sequence': sequence,
                'cofactors': set()
            }
        all_sequences[protein_id]['cofactors'].add(cofactor)
    
    print(f"  Retrieved {len(sequences)} sequences")
    time.sleep(1)

print(f"\nTotal unique proteins retrieved: {len(all_sequences)}")

Fetching sequences from NCBI...

Fetching sequences for Cu-binding enzymes (500 proteins)...
  Retrieved 321 sequences
Fetching sequences for ATP-binding enzymes (500 proteins)...
    Batch 27 failed, retrying individually...
    ⚠ Skipped 2 invalid/inaccessible IDs
  Retrieved 498 sequences
Fetching sequences for B12-binding enzymes (500 proteins)...
  Retrieved 486 sequences
Fetching sequences for CoA-binding enzymes (500 proteins)...
  Retrieved 448 sequences
Fetching sequences for NAD-binding enzymes (500 proteins)...
  Retrieved 492 sequences

Total unique proteins retrieved: 2241


In [7]:
# Create DataFrame with binary cofactor labels

print("Creating DataFrame with binary cofactor labels...\n")

data_rows = []

for protein_id, data in all_sequences.items():
    row = {
        'header': data['header'],
        'sequence': data['sequence']
    }
    
    # Create binary labels for each cofactor
    for cofactor in cofactors:
        row[cofactor] = 1 if cofactor in data['cofactors'] else 0
    
    data_rows.append(row)

df = pd.DataFrame(data_rows)

# Reorder columns to match format: header, sequence, Cu, Fe, Zn, Mn, Mg
column_order = ['header', 'sequence'] + cofactors
df = df[column_order]

print(f"DataFrame created with {len(df)} proteins")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

Creating DataFrame with binary cofactor labels...

DataFrame created with 2241 proteins

Columns: ['header', 'sequence', 'Cu', 'ATP', 'B12', 'CoA', 'NAD']

DataFrame shape: (2241, 7)

First few rows:
                                              header  \
0  NP_013752.1 copper chaperone CCS1 [Saccharomyc...   
1       pdb|1PF3|A Chain A, Blue copper oxidase cueO   
2  WP_415607860.1 copper homeostasis protein CutC...   
3  WP_289940195.1 copper homeostasis protein CutC...   
4      CAN5492343.1 copper oxidase [soil metagenome]   

                                            sequence  Cu  ATP  B12  CoA  NAD  
0  MTTNDTYEATYAIPMHCENCVNDIKACLKNVPGINSLNFDIEQQIM...   1    0    0    0    0  
1  AERPTLPIPDLLTTDARNRIQLTIGAGQSTFGGKTATTWGYNGNLL...   1    0    1    1    1  
2  MTIKIKEAAVDSADRAQEMIARGANRIELNARLDLGGITPDTRTII...   1    0    0    0    0  
3  MARRLGSERRTVTTAADPASLAEKSSTRARALVEICVDDLAGVLAA...   1    0    0    0    0  
4  MRAAGATMIGAAAVSRAAAASLPEAETRSSPAMRPPPSPPNGRPFH...   1    0    0  

In [8]:
# Display summary statistics

print("="*80)
print("SUMMARY STATISTICS")
print("="*80)

print(f"\nTotal proteins: {len(df)}")
print(f"\nCofactor usage:")
for cofactor in cofactors:
    count = df[cofactor].sum()
    print(f"  {cofactor}: {count} proteins ({count/len(df)*100:.1f}%)")

print(f"\nSequence lengths:")
df['seq_length'] = df['sequence'].str.len()
print(f"  Min: {df['seq_length'].min()}")
print(f"  Max: {df['seq_length'].max()}")
print(f"  Mean: {df['seq_length'].mean():.1f}")
print(f"  Median: {df['seq_length'].median():.0f}")

SUMMARY STATISTICS

Total proteins: 2241

Cofactor usage:
  Cu: 321 proteins (14.3%)
  ATP: 498 proteins (22.2%)
  B12: 486 proteins (21.7%)
  CoA: 448 proteins (20.0%)
  NAD: 492 proteins (22.0%)

Sequence lengths:
  Min: 31
  Max: 4951
  Mean: 512.7
  Median: 440


In [9]:
# Export to CSV file

output_filename = "ncbi_enzyme_cofactor_data.csv"

# Export with exact column order
df_export = df[column_order].copy()
df_export.to_csv(output_filename, index=False)

print("="*80)
print(f"✓ Data exported to: {output_filename}")
print("="*80)
print(f"\nFile contains {len(df_export)} proteins with columns:")
print(f"  {', '.join(df_export.columns)}")
print(f"\nFile size: {len(df_export)} rows x {len(df_export.columns)} columns")

✓ Data exported to: ncbi_enzyme_cofactor_data.csv

File contains 2241 proteins with columns:
  header, sequence, Cu, ATP, B12, CoA, NAD

File size: 2241 rows x 7 columns


In [10]:
# Display sample data

print("="*80)
print("SAMPLE DATA (first 5 proteins)")
print("="*80)

# Display with truncated sequences for readability
df_display = df.copy()
df_display['sequence_preview'] = df_display['sequence'].str[:50] + '...'
display_cols = ['header', 'sequence_preview'] + cofactors

print(df_display[display_cols].head(5).to_string(index=False))

SAMPLE DATA (first 5 proteins)
                                                                              header                                      sequence_preview  Cu  ATP  B12  CoA  NAD
                  NP_013752.1 copper chaperone CCS1 [Saccharomyces cerevisiae S288C] MTTNDTYEATYAIPMHCENCVNDIKACLKNVPGINSLNFDIEQQIMSVES...   1    0    0    0    0
                                        pdb|1PF3|A Chain A, Blue copper oxidase cueO AERPTLPIPDLLTTDARNRIQLTIGAGQSTFGGKTATTWGYNGNLLGPAV...   1    0    1    1    1
WP_415607860.1 copper homeostasis protein CutC, partial [Leuconostoc falkenbergense] MTIKIKEAAVDSADRAQEMIARGANRIELNARLDLGGITPDTRTIIDTLT...   1    0    0    0    0
           WP_289940195.1 copper homeostasis protein CutC [Microbacterium testaceum] MARRLGSERRTVTTAADPASLAEKSSTRARALVEICVDDLAGVLAAERAG...   1    0    0    0    0
                                       CAN5492343.1 copper oxidase [soil metagenome] MRAAGATMIGAAAVSRAAAASLPEAETRSSPAMRPPPSPPNGRPFHPVVT...   1    0    0  